### Import & Config

In [32]:
import requests
import json
import time
import os
import pandas as pd
from dotenv import load_dotenv

# Load API credentials from .env file
load_dotenv(override=True)
API_KEY = os.getenv("RAINFOREST_API_KEY_2")
print(API_KEY)
# API configuration
BASE_URL = "https://api.rainforestapi.com/request"

OUTPUT_DIR = "../data/api_raw/products"
os.makedirs(OUTPUT_DIR, exist_ok=True)

4100830CBAC44806AC4FBED0544B3457


In [33]:
print(os.getenv("RAINFOREST_API_KEY_2"))


4100830CBAC44806AC4FBED0544B3457


### Load ASINs

In [23]:
with open('../data/processed/top_asins.json', 'r') as f:
    asins = json.load(f)

print(f"ASINs to query: {len(asins)}")
asins

ASINs to query: 32


['B0D8W1YVBX',
 'B08KT2Z93D',
 'B09541P9WH',
 'B00AHAWWO0',
 'B0DBF65JYY',
 'B0DPHQRLJC',
 'B0BK2SC18T',
 'B091NJQ29P',
 'B0113UZJE2',
 'B0CJ1B6D6S',
 'B07PZF3QS3',
 'B0CTJGJL2T',
 'B0F2TB2MMP',
 'B07HRCDDL1',
 'B0C3QZ7SNF',
 'B0CQMRKRV5',
 'B0DGHMNQ5Z',
 'B0FC5FJZ9Z',
 'B0FC6S2R7K',
 'B0FLXY2CBC',
 'B0B72DBKVF',
 'B01FWAZEIU',
 'B0F43VY6TC',
 'B09B2SBHQK',
 'B07HJSHT8P',
 'B088H5DWV3',
 'B0009KF59W',
 'B0CG2LW9RN',
 'B08JGNRJ8P',
 'B08P286Q26',
 'B0C9BLDRYS',
 'B09GNQD678']

In [34]:
test_asin = asins[0]

params = {
    "api_key":       API_KEY,
    "type":          "product",
    "asin":          test_asin,
    "amazon_domain": "amazon.com",
}

response = requests.get(BASE_URL, params=params)
print("Status:", response.status_code)
print(response.json()['request_info'])

Status: 200
{'success': True, 'credits_used': 1, 'credits_remaining': 99, 'credits_used_this_request': 1}


### Function with Checkpoint

In [35]:
def fetch_product(asin: str) -> None:
    # fetch full product details by asin and save raw JSON file in api_raw/products
    filepath = os.path.join(OUTPUT_DIR, f"{asin}.json")
    
    if os.path.exists(filepath):
        print(f"[SKIP] {asin}")
        return

    params = {
        "api_key":       API_KEY,
        "type":          "product",
        "asin":          asin,
        "amazon_domain": "amazon.com",
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print(f"[ERROR] {asin} — status {response.status_code}")
        return

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, ensure_ascii=False, indent=2)

    print(f"[OK] {asin}")
    time.sleep(3)

### Execution

In [36]:
for asin in asins:
    fetch_product(asin)

print("Done.")

[OK] B0D8W1YVBX
[OK] B08KT2Z93D
[OK] B09541P9WH
[OK] B00AHAWWO0
[OK] B0DBF65JYY
[OK] B0DPHQRLJC
[OK] B0BK2SC18T
[OK] B091NJQ29P
[OK] B0113UZJE2
[OK] B0CJ1B6D6S
[OK] B07PZF3QS3
[OK] B0CTJGJL2T
[OK] B0F2TB2MMP
[OK] B07HRCDDL1
[OK] B0C3QZ7SNF
[OK] B0CQMRKRV5
[OK] B0DGHMNQ5Z
[OK] B0FC5FJZ9Z
[OK] B0FC6S2R7K
[OK] B0FLXY2CBC
[OK] B0B72DBKVF
[OK] B01FWAZEIU
[OK] B0F43VY6TC
[OK] B09B2SBHQK
[OK] B07HJSHT8P
[OK] B088H5DWV3
[OK] B0009KF59W
[OK] B0CG2LW9RN
[OK] B08JGNRJ8P
[OK] B08P286Q26
[OK] B0C9BLDRYS
[OK] B09GNQD678
Done.


### This is the request test with 1 product

In [30]:
# try 1 request for product
params = {
    'api_key'       : API_KEY,
    'type'          :'product',
    'asin'          : asins[0],
    'amazon_domain' :'amazon.com'
}

response = requests.get(BASE_URL, params=params)
print("Status:", response.status_code)


if response.status_code == 200:
    data = response.json()
    product = data.get('product', {})
            
    # Categories
    categories = product.get('categories', [])

    # Buybox
    buybox = product.get('buybox_winner', {})
    fulfillment = buybox.get('fulfillment', {})
    seller = fulfillment.get('third_party_seller', {})

    # Best Sellers Rank
    bsr = product.get('bestsellers_rank', [])

    brand      = product.get('brand')
    subcategory  = categories[-1].get('name') if categories else None
    image      = product.get('main_image', {}).get('link')
    is_fba     = fulfillment.get('is_fulfilled_by_amazon')
    bsr_main     = bsr[0].get('rank') if len(bsr) > 0 else None
    bsr_sub      = bsr[1].get('rank') if len(bsr) > 1 else None
    seller_name = seller.get('name')
    print("\n--- First product ---")
    print("ASIN:    ", asins[0])
    print("brand:   ", brand)
    print('seller name', seller_name)
    print("is_fba:  ",is_fba)
    print("image:   ", image)
    print("subcategory:", subcategory)
    print("bsr_main: ",bsr_main)
    print('bsr_sub', bsr_sub)
        
else:
    print(f"Request failed — status {response.status_code}")
    print(response.text)

Status: 402
Request failed — status 402
{"request_info":{"success":false,"message":"Your account has been temporarily suspended as our systems detected multiple Free Trial accounts being active, or a disposable email address being used during signup. We allow one Free Trial per organisation and to ensure the integrity of Free Trials we do not permit disposable email addresses to be used. We'd love to welcome you as a customer so we'd kindly invite you to consider one of our paid plans or sign up for a Free Trial using a valid email address. This suspension will automatically be removed when you subscribe to a Plan."}}
